# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


## Hizli Akis

Bu notebook secilen tek adapter icin training, OOD kalibrasyonu, final evaluation ve export akisini sirayla calistirir.

- Genelde sadece `ADAPTER_KEY` degistirin; crop, part, dataset, OOD ve OE yollari bu anahtardan acik sekilde set edilir.
- Runtime dataset beklentisi: `data/prepared_runtime_datasets/<adapter_key>/continual|val|test|ood|oe`.
- `ood` final kanit, `oe` sadece training yardimci negatifleridir; ikisini karistirmayin.
- Push veya publish son adimda bozulursa training bittiyse run klasorunu kurtarmayi deneyin, bastan egitime donmeyin.


In [1]:
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK2_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)

def _build_repo_access_url(repo_url, token):
    token = str(token or '').strip()
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    clone_url = _build_repo_access_url(REPO_URL, os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN'))
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', clone_url, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK2_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


[BOOTSTRAP] GitHub token found.
[BOOTSTRAP] HuggingFace token found.
[BOOTSTRAP] Repo root: /content/bitirmeprojesi
[BOOTSTRAP] Installing requirements_colab.txt...
[BOOTSTRAP] Notebook 2: Continual Adapter Training bootstrap complete.

BOOTSTRAP STATUS
Status: ok
Repo Root: /content/bitirmeprojesi
In Colab: True
GitHub Token: ✓
HuggingFace Token: ✓

[SONRAKI] Parametre ve adapter secimi hucresine gecin; model erisimi ayrica access-check hucresinde dogrulanacak.


In [2]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())


[BOOTSTRAP] run_id=tomato_unspecified_2026-05-11_07-41-08 crop=tomato part=unspecified


In [3]:
# Adapter secimi: genelde sadece bunu degistir.
ADAPTER_KEY = "strawberry__fruit"  # grape__fruit, grape__leaf, strawberry__fruit, strawberry__leaf, tomato__fruit, tomato__leaf

# Bayesian optimizasyon default olarak acik: onceki run'lardan ayni adapter icin parametre onerisi uygular.
ENABLE_BAYESIAN_OPTIMIZATION = True

# Adapter bazli gorunur defaultlar. ADAPTER_KEY hangi bloğun kullanilacagini secer.
ADAPTER_RECS = {
    "grape__fruit": {"crop": "grape", "part": "fruit", "ood": "data/prepared_runtime_datasets/grape__fruit/ood", "oe": "data/prepared_runtime_datasets/grape__fruit/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False, "overrides": {"EPOCHS": 32, "BATCH_SIZE": 48, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0}},
    "grape__leaf": {"crop": "grape", "part": "leaf", "ood": "data/prepared_runtime_datasets/grape__leaf/ood", "oe": "data/prepared_runtime_datasets/grape__leaf/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False, "overrides": {"EPOCHS": 28, "BATCH_SIZE": 64, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.12, "OOD_FACTOR": 3.0}},
    "strawberry__fruit": {"crop": "strawberry", "part": "fruit", "ood": "data/prepared_runtime_datasets/strawberry__fruit/ood", "oe": "data/prepared_runtime_datasets/strawberry__fruit/oe", "oe_enabled": True, "oe_w": 0.10, "allow_under_min": True, "overrides": {"EPOCHS": 36, "BATCH_SIZE": 32, "LEARNING_RATE": 8e-5, "LORA_R": 16, "LORA_ALPHA": 16, "LORA_DROPOUT": 0.18, "OOD_FACTOR": 3.0}},
    "strawberry__leaf": {"crop": "strawberry", "part": "leaf", "ood": "data/prepared_runtime_datasets/strawberry__leaf/ood", "oe": "data/prepared_runtime_datasets/strawberry__leaf/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 22, "BATCH_SIZE": 96, "LEARNING_RATE": 1.5e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0}},
    "tomato__fruit": {"crop": "tomato", "part": "fruit", "ood": "data/prepared_runtime_datasets/tomato__fruit/ood", "oe": "data/prepared_runtime_datasets/tomato__fruit/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 30, "BATCH_SIZE": 64, "LEARNING_RATE": 8e-5, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0}},
    "tomato__leaf": {"crop": "tomato", "part": "leaf", "ood": "data/prepared_runtime_datasets/tomato__leaf/ood", "oe": "data/prepared_runtime_datasets/tomato__leaf/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 20, "BATCH_SIZE": 96, "LEARNING_RATE": 1.2e-4, "LORA_R": 32, "LORA_ALPHA": 32, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0}},
}

# Adapterden bagimsiz gorunur defaultlar.
DEFAULT_RUNTIME_PARAMS = {
    "ENABLE_BAYESIAN_OPTIMIZATION": ENABLE_BAYESIAN_OPTIMIZATION,
    "WEIGHT_DECAY": 0.01,
    "LOSS_NAME": "logitnorm",
    "LOGITNORM_TAU": 1.0,
    "MIXED_PRECISION": "bf16",
    "GRAD_ACCUM_STEPS": 1,
    "MAX_GRAD_NORM": 1.0,
    "LABEL_SMOOTHING": 0.0,
    "SCHEDULER_NAME": "cosine",
    "SCHEDULER_WARMUP_RATIO": 0.1,
    "SCHEDULER_MIN_LR": 1e-6,
    "EARLY_STOPPING_PATIENCE": 10,
    "EARLY_STOPPING_MIN_DELTA": 0.0,
    "DETERMINISTIC": False,
    "SEED": 42,
    "RANDAUGMENT_NUM_OPS": 2,
    "RANDAUGMENT_MAGNITUDE": 7,
    "NUM_WORKERS": 12,
    "PREFETCH": 8,
    "PIN_MEMORY": True,
    "USE_CACHE": True,
    "CACHE_TRAIN_SPLIT": True,
    "VALIDATION_EVERY_N_EPOCHS": 1,
    "CHECKPOINT_EVERY_N_STEPS": 250,
    "CHECKPOINT_ON_EXCEPTION": True,
    "STDOUT_BATCH_INTERVAL": 12,
    "RESUME_MODE": "fresh",
    "AUTO_DISCONNECT_RUNTIME": True,
    "AUTO_DISCONNECT_GRACE_SECONDS": 20,
    "AUTO_PUSH_TO_GITHUB": True,
    "AUTO_PUSH_REMOTE_NAME": "origin",
    "AUTO_PUSH_BRANCH": None,
}

# Sadece bu run icin defaultlari ezmek istersen buraya yaz.
MANUAL_PARAM_OVERRIDES = {}

with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    from scripts.notebook_helpers.cell_script_runner import run_cell_script
    run_cell_script('nb2_cell04_parameter_resolution.py', globals())


[ADAPTER_SELECTED] key=strawberry__fruit crop=strawberry part=fruit dataset=strawberry__fruit ood=data/prepared_runtime_datasets/strawberry__fruit/ood oe_enabled=True oe=data/prepared_runtime_datasets/strawberry__fruit/oe run_id=strawberry_fruit_2026-05-11_07-41-08
[TRAINING_PLAN] epochs=36 batch=32 lr=8e-05 lora_r=16 allow_under_min=True validation_every=1
[SONRAKI] Erisim kontrolu -> dataset validation -> engine init -> training -> calibration -> save/eval hucresini sirayla calistirin.
[BAYES] rank=1 ile 12 parametre onerisi yuklendi.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=strawberry epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=True
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=strawberry__fruit
[OOD] ood_root=data/prepared_runtime_datasets/strawberry__fruit/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runtime_datasets/strawberry__fruit/oe ask_for_oe_root=False enabled=True loss_weight=0.1
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=1 cache_train=True aug=randaugment randaug=2/7 
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [4]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.


In [5]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


[OOD] explicit ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__fruit/ood
[OE] explicit oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__fruit/oe
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__fruit classes=5: ['strawberry_anthracnose_fruit', 'strawberry_gray_mold_fruit', 'strawberry_healthy_fruit', 'strawberry_powdery_mildew_fruit', 'strawberry_unripe_fruit']
[DATASET][CHECK] scale=small continual=688 val=223 test=223 ood=219 classes=5
[DATASET][WARN] Manifest rows include 5 eval-quality-risk item(s); prefer cautious regularization.
[DATASET][BLOCK] Supported classes remain below the production minimum of 100 images/class: strawberry_anthracnose_fruit=88. Enable ALLOW_UNDER_MIN_TRAINING manually only for a research-only run.
[PARAMS] Manual overrides uygulandi: {'EPOCHS': 36, 'BATCH_SIZE': 32, 'LEARNING_RATE': 8e-05, 'LORA_R': 16, 'LORA_ALPHA': 16, 'LORA_DROPOUT': 0.18, 'OOD_FACTOR': 3.0, '

In [6]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


[OOD] selected ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__fruit/ood
[OE] selected oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__fruit/oe
[CLASSES] ['strawberry_anthracnose_fruit', 'strawberry_gray_mold_fruit', 'strawberry_healthy_fruit', 'strawberry_powdery_mildew_fruit', 'strawberry_unripe_fruit']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=0 unmatched=5
[CLASSES] taxonomy-unmatched classes kept: ['strawberry_anthracnose_fruit', 'strawberry_gray_mold_fruit', 'strawberry_healthy_fruit', 'strawberry_powdery_mildew_fruit', 'strawberry_unripe_fruit']
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature": 1.0, "energy_temperature_mode": "auto", "energy_temperature_range": [0.5, 3

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=10,232,073  classes=5


In [7]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.


In [8]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


[TRAIN] epochs=36 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/21 loss=0.0000 lr=0.000080 speed=214.5s/s elapsed=5s eta=285s
[CKPT] epoch_end epoch=1 step=21
[EPOCH] 1/36: train_loss=1.6302 val_loss=1.6497 val_acc=0.1345 macro_f1=0.0474 * BEST
[TRAIN] epoch=2 batch=12/21 loss=0.0000 lr=0.000080 speed=215.0s/s elapsed=14s eta=311s
[EPOCH] 2/36: train_loss=1.6004 val_loss=1.6862 val_acc=0.1031 macro_f1=0.0374
[TRAIN] epoch=3 batch=12/21 loss=0.0000 lr=0.000079 speed=206.5s/s elapsed=20s eta=256s
[CKPT] epoch_end epoch=3 step=63
[EPOCH] 3/36: train_loss=1.5853 val_loss=1.3366 val_acc=0.5291 macro_f1=0.3216 * BEST
[TRAIN] epoch=4 batch=12/21 loss=0.0000 lr=0.000078 speed=212.8s/s elapsed=28s eta=257s
[CKPT] epoch_end epoch=4 step=84
[EPOCH] 4/36: train_loss=1.4278 val_loss=1.0691 val_acc=0.7040 macro_f1=0.5824 * BEST
[TRAIN] epoch=5 batch=12/21 loss=0.0000 lr=0.000077 speed=205.1s/s elapsed=37s eta=252s
[CKPT] epoch_end epoch=5 step=105
[EPOCH] 5/36: train_loss=1.2371 val_loss=0.

In [9]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


[OOD] Kalibrasyon tamamlandi. classes=5 version=1


In [10]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


Cell 8: adapter save started
Kaydedilen adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter
 - outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/strawberry/fruit/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/strawberry_fruit_2026-05-11_07-41-08/artifacts/adapter_export/strawberry/fruit/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
# Auto-extracted from colab_notebooks/2_train_continual_sd_lora_adapter.ipynb cell 12.
# Keep notebook execute-only cells thin; edit behavior here.

with TELEMETRY.capture_cell_output("Cell 9: Final Evaluation"):
    import json
    from datetime import datetime, timezone
    from scripts.colab_notebook_helpers import (
        merge_training_summary_fields,
        notebook_artifact_root,
        persist_production_readiness_artifact,
        persist_validation_artifacts,
        print_notebook_optimization_campaign_status,
        resolve_notebook_optimization_campaign,
        summarize_notebook_optimization_campaign,
    )
    from src.adapter.independent_crop_adapter import IndependentCropAdapter
    from src.training.services.ood_benchmark import run_leave_one_class_out_benchmark
    from src.training.validation import evaluate_model_with_artifact_metrics

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Once engine init hucresi calistirilmali.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    effective_params = dict(STATE.get("effective_params") or {})
    test_loader = loaders.get("test")
    if test_loader is None or len(test_loader.dataset) == 0:
        raise RuntimeError("Test loader bos. Final degerlendirme held-out test split ile yapilmali.")

    trainer = adapter._trainer
    if trainer is None:
        raise RuntimeError("Trainer hazir degil.")

    notebook_config = json.loads(json.dumps(BASE_CONFIG))
    notebook_config.setdefault("training", {})["continual"] = json.loads(json.dumps(STATE["continual_config"]))
    evaluation_cfg = notebook_config.get("training", {}).get("continual", {}).get("evaluation", {})
    artifact_root = notebook_artifact_root(ROOT)

    trainer.adapter_model.eval()
    trainer.classifier.eval()
    trainer.fusion.eval()

    classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda item: item[1])]
    ood_loader = loaders.get("ood")

    def _evaluate_split(split_name: str, loader, *, artifact_subdir: str, label: str):
        evaluation = evaluate_model_with_artifact_metrics(trainer, loader, ood_loader=ood_loader)
        if evaluation is None:
            raise RuntimeError("Degerlendirme ornegi bulunamadi.")
        artifacts = persist_validation_artifacts(
            root=ROOT,
            y_true=evaluation.y_true,
            y_pred=evaluation.y_pred,
            classes=classes,
            telemetry=TELEMETRY,
            artifact_subdir=artifact_subdir,
            require_ood=bool(evaluation_cfg.get("require_ood_for_gate", True)),
            emit_metric_gate=bool(evaluation_cfg.get("emit_ood_gate", True)),
            ood_labels=evaluation.ood_labels,
            ood_scores=evaluation.ood_scores,
            ood_scores_by_method=evaluation.ood_scores_by_method,
            sure_ds_f1=evaluation.sure_ds_f1,
            conformal_empirical_coverage=evaluation.conformal_empirical_coverage,
            conformal_avg_set_size=evaluation.conformal_avg_set_size,
            ood_type_breakdown=evaluation.ood_type_breakdown,
            prediction_rows=evaluation.prediction_rows,
            context={
                "crop_name": CROP_NAME,
                "part_name": PART_NAME,
                "run_id": RUN_ID,
                "split_name": split_name,
                **evaluation.context,
            },
        )
        metrics = artifacts["metric_gate"]["metrics"]
        extras = []
        if metrics.get("ood_auroc") is not None:
            extras.append(f"ood_auroc={float(metrics['ood_auroc']):.4f}")
        if metrics.get("sure_ds_f1") is not None:
            extras.append(f"sure_ds_f1={float(metrics['sure_ds_f1']):.4f}")
        if metrics.get("conformal_empirical_coverage") is not None:
            extras.append(f"conformal_cov={float(metrics['conformal_empirical_coverage']):.4f}")
        suffix = " " + " ".join(extras) if extras else ""
        accuracy = float(artifacts["report_dict"].get("accuracy", 0.0))
        print(f"[{label}] ornek={len(evaluation.y_true)} sinif={len(classes)} accuracy={accuracy:.4f}{suffix}")
        return artifacts

    results = {}
    val_loader = loaders.get("val")
    if val_loader is not None and len(val_loader.dataset) > 0:
        results["validation"] = _evaluate_split(
            "val",
            val_loader,
            artifact_subdir="validation",
            label="DOGRULAMA (referans)",
        )

    results["test"] = _evaluate_split(
        "test",
        test_loader,
        artifact_subdir="test",
        label="TEST (ayrilmis)",
    )
    selected_split = "test" if "test" in results else "validation"
    selected_artifacts = results[selected_split]
    real_ood_present = ood_loader is not None and len(ood_loader.dataset) > 0
    ood_evidence_source = "real_ood_split" if real_ood_present else "unavailable"
    ood_evidence_metrics = dict(selected_artifacts["metric_gate"]["metrics"]) if real_ood_present else {}
    benchmark_summary = {}
    if (
        not real_ood_present
        and str(evaluation_cfg.get("ood_fallback_strategy", "held_out_benchmark")) == "held_out_benchmark"
        and bool(evaluation_cfg.get("ood_benchmark_auto_run", True))
    ):
        print("[OOD] Gercek OOD split bulunamadi; held-out benchmark fallback calisiyor...")
        benchmark_summary = run_leave_one_class_out_benchmark(
            crop_name=CROP_NAME,
            class_names=classes,
            loaders=loaders,
            config=notebook_config,
            device=DEVICE,
            artifact_root=artifact_root,
            adapter_factory=IndependentCropAdapter,
            run_id=RUN_ID,
            num_epochs=int(effective_params["EPOCHS"]),
            telemetry=TELEMETRY,
            emit_event=lambda event_type, payload: TELEMETRY.emit_event(event_type, payload, phase="evaluation"),
            min_classes=int(evaluation_cfg.get("ood_benchmark_min_classes", 3)),
        )
        ood_evidence_source = "held_out_benchmark"
        ood_evidence_metrics = dict(benchmark_summary.get("metrics", {}))

    readiness = persist_production_readiness_artifact(
        root=ROOT,
        classification_metric_gate=selected_artifacts.get("metric_gate"),
        classification_split=selected_split,
        ood_evidence_source=ood_evidence_source,
        ood_metrics=ood_evidence_metrics,
        context={
            "run_id": RUN_ID,
            "crop_name": CROP_NAME,
            "part_name": PART_NAME,
            "classification_split": selected_split,
            "ood_benchmark_status": benchmark_summary.get("status"),
            "ood_benchmark_passed": benchmark_summary.get("passed"),
        },
        require_ood=bool(evaluation_cfg.get("require_ood_for_gate", True)),
        telemetry=TELEMETRY,
    )

    STATE["evaluation_artifacts"] = results
    STATE["ood_benchmark"] = benchmark_summary
    STATE["production_readiness"] = readiness["payload"]
    # Ensure matplotlib is lazily loaded before using plt
    _ensure_matplotlib()
    plt.close("all")
    print(
        f"[OOD] kanit={readiness['payload'].get('ood_evidence_source', 'unavailable')} "
        f"durum={readiness['payload'].get('status', 'failed')} gecti={bool(readiness['payload'].get('passed', False))}"
    )

    TELEMETRY.update_latest(
        {
            "phase": "evaluation_complete",
            "evaluation_splits": sorted(results.keys()),
            "ood_evidence_source": readiness["payload"].get("ood_evidence_source"),
            "production_readiness": readiness["payload"].get("status"),
        }
    )
    print("[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.")

from scripts.colab_notebook_helpers import build_notebook_completion_report, maybe_auto_disconnect_colab_runtime, merge_training_summary_fields, print_notebook_optimization_campaign_status, resolve_notebook_optimization_campaign, summarize_notebook_optimization_campaign

REPO_RUN_EXPORTS = save_run_outputs_to_repo()
effective_params = dict(STATE.get("effective_params") or {})
resolved_dataset_root = Path(STATE.get("runtime_dataset_root") or RUNTIME_DATASET_ROOT)
resolved_dataset_key = str(STATE.get("runtime_dataset_key") or STATE.get("selected_dataset_name") or DATASET_NAME or "").strip()
if resolved_dataset_key:
    optimization_campaign_mode = "continue" if bool(effective_params.get("ENABLE_BAYESIAN_OPTIMIZATION", True)) else "disabled"
    STATE["optimization_campaign"] = resolve_notebook_optimization_campaign(
        root=ROOT,
        runtime_dataset_root=resolved_dataset_root,
        dataset_key=resolved_dataset_key,
        crop_name=CROP_NAME,
        part_name=PART_NAME,
        backbone_model_name=str(dict(BASE_CONFIG or {}).get("training", {}).get("continual", {}).get("backbone", {}).get("model_name", "")),
        notebook_parameters=effective_params,
        mode=optimization_campaign_mode,
        telemetry=TELEMETRY,
    )
    STATE["recommendation_report"] = summarize_notebook_optimization_campaign(STATE["optimization_campaign"])
    STATE["recommendation_decision"] = str(STATE["optimization_campaign"].get("status", "disabled"))
    print_notebook_optimization_campaign_status(STATE["optimization_campaign"], print_fn=print)
notebook_export_result = export_current_colab_notebook(REPO_NOTEBOOK_OUTPUT_PATH)

extra_entries = [
    {
        "path": build_adapter_bundle_root(REPO_OUTPUT_DIR, CROP_NAME, PART_NAME) / "continual_sd_lora_adapter",
        "category": "adapter_export",
        "priority": "high",
        "title_tr": "Repo mirror adapter export klasoru",
        "description_tr": "Repo mirror icindeki adapter export klasoru.",
        "reader_goal": "Export edilen adapter klasorunu bulmak",
        "generated_by": "notebook_2",
        "decision_importance": "deploy_handoff",
        "read_order": 70,
    },
    {
        "path": REPO_NOTEBOOK_OUTPUT_PATH,
        "category": "adapter_export",
        "priority": "medium",
        "title_tr": "Calistirilmis notebook exportu",
        "description_tr": "Bu kosuda calisan notebook'un kaydedilmis kopyasi.",
        "reader_goal": "Notebook'u ayni ciktiyla tekrar incelemek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 71,
    },
    {
        "path": REPO_TELEMETRY_DIR / "events.jsonl",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Telemetry event logu",
        "description_tr": "Notebook olayi bazli telemetry kaydi.",
        "reader_goal": "Notebook akisini olay bazinda incelemek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 80,
    },
    {
        "path": REPO_TELEMETRY_DIR / "runtime.log",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Runtime logu",
        "description_tr": "Notebook runtime boyunca yazilan metin logu.",
        "reader_goal": "Calisma sirasindaki log ciktilarini okumak",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 81,
    },
    {
        "path": REPO_TELEMETRY_DIR / "latest_status.json",
        "category": "logs_and_checkpoints",
        "priority": "low",
        "title_tr": "Son durum ozeti",
        "description_tr": "Notebook'un son durum snapshot'i.",
        "reader_goal": "Kosunun son durumunu hizli kontrol etmek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 82,
    },
    {
        "path": REPO_TELEMETRY_DIR / "summary.json",
        "category": "logs_and_checkpoints",
        "priority": "high",
        "title_tr": "Telemetry ozeti",
        "description_tr": "Notebook final ozet dosyasi.",
        "reader_goal": "Notebook final ozetini okumak",
        "generated_by": "notebook_2",
        "decision_importance": "run_overview",
        "read_order": 83,
    },
    {
        "path": REPO_CHECKPOINT_STATE_DIR / "best_checkpoint.json",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Best checkpoint manifesti",
        "description_tr": "En iyi checkpoint'in repo mirror manifesti.",
        "reader_goal": "Hangi checkpoint secildigini gormek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 84,
    },
    {
        "path": REPO_CHECKPOINT_STATE_DIR / "latest_checkpoint.json",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Latest checkpoint manifesti",
        "description_tr": "Son checkpoint manifesti.",
        "reader_goal": "Checkpoint akisini gormek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 85,
    },
    {
        "path": REPO_CHECKPOINT_STATE_DIR / "checkpoint_index.json",
        "category": "logs_and_checkpoints",
        "priority": "low",
        "title_tr": "Checkpoint indexi",
        "description_tr": "Mirror edilen checkpoint manifest listesi.",
        "reader_goal": "Checkpoint kayitlarini toplu gormek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 86,
    },
]
summary_payload = merge_training_summary_fields(
    root=ROOT,
    telemetry=TELEMETRY,
    payload={
        "recommendation": {
            "decision": str(STATE.get("recommendation_decision") or "pending"),
            "change_count": int((STATE.get("recommendation_report") or {}).get("change_count", 0)),
            "manual_overrides": dict(MANUAL_PARAM_OVERRIDES or {}),
        },
        "run_id": RUN_ID,
        "run_label": RUN_ID,
        "crop_name": CROP_NAME,
        "part_name": PART_NAME,
        "notebook_surface": "2_train_continual_sd_lora_adapter.ipynb",
        "dataset_roots": {
            "runtime_dataset_root": RUNTIME_DATASET_ROOT,
            "runtime_dataset_key": str(STATE.get("runtime_dataset_key") or ""),
            "selected_dataset_name": str(STATE.get("selected_dataset_name") or ""),
            "selected_runtime_dataset_root": str(STATE.get("selected_dataset_root") or ""),
            "resolved_ood_root": str(STATE.get("resolved_ood_root") or ""),
            "resolved_runtime_dataset_root": str(STATE.get("runtime_dataset_root") or ""),
        },
        "notebook_parameters": {
            "epochs": (STATE.get("effective_params") or {}).get("EPOCHS"),
            "batch_size": (STATE.get("effective_params") or {}).get("BATCH_SIZE"),
            "learning_rate": (STATE.get("effective_params") or {}).get("LEARNING_RATE"),
            "lora_r": (STATE.get("effective_params") or {}).get("LORA_R"),
            "ood_factor": (STATE.get("effective_params") or {}).get("OOD_FACTOR"),
            "mixed_precision": (STATE.get("effective_params") or {}).get("MIXED_PRECISION"),
            "num_workers": (STATE.get("effective_params") or {}).get("NUM_WORKERS"),
            "checkpoint_every_n_steps": (STATE.get("effective_params") or {}).get("CHECKPOINT_EVERY_N_STEPS"),
            "enable_bayesian_optimization": bool(
                (STATE.get("effective_params") or {}).get("ENABLE_BAYESIAN_OPTIMIZATION", True)
            ),
        },
        "export_paths": {
            "repo_run_dir": str(REPO_RUN_DIR),
            "repo_output_dir": str(REPO_OUTPUT_DIR),
            "repo_telemetry_dir": str(REPO_TELEMETRY_DIR),
            "repo_checkpoint_state_dir": str(REPO_CHECKPOINT_STATE_DIR),
            "executed_notebook_path": str(notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH),
            "adapter_export_dir": str(STATE.get("adapter_export_dir") or ""),
        },
        "access_check": STATE.get("access_report", {}),
        "readiness_summary": {
            "status": (STATE.get("production_readiness") or {}).get("status"),
            "passed": (STATE.get("production_readiness") or {}).get("passed"),
            "ood_evidence_source": (STATE.get("production_readiness") or {}).get("ood_evidence_source"),
        },
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    extra_entries=extra_entries,
)
TELEMETRY.merge_summary_metadata(
    {
        "access_check": STATE.get("access_report", {}),
        "repo_paths": {
            "repo_run_dir": str(REPO_RUN_DIR),
            "repo_output_dir": str(REPO_OUTPUT_DIR),
            "repo_telemetry_dir": str(REPO_TELEMETRY_DIR),
            "repo_checkpoint_state_dir": str(REPO_CHECKPOINT_STATE_DIR),
        },
        "training_summary": summary_payload,
    }
)
TELEMETRY.close(
    {
        "status": "ok",
        "evaluation_splits": sorted((STATE.get("evaluation_artifacts") or {}).keys()),
        "cell_outputs_dir": str(TELEMETRY.artifacts_dir / "cell_outputs"),
        "repo_run_dir": str(REPO_RUN_DIR),
        "run_label": RUN_ID,
    }
)
REPO_RUN_EXPORTS = save_run_outputs_to_repo()
for key in sorted(REPO_RUN_EXPORTS):
    print(f"[RUNS] {key} -> {REPO_RUN_EXPORTS[key]}")
print(f"[RUNS] notebook -> {REPO_NOTEBOOK_OUTPUT_PATH}")
effective_params = dict(STATE.get("effective_params") or {})
if bool(effective_params.get("AUTO_PUSH_TO_GITHUB", AUTO_PUSH_TO_GITHUB)):
    try:
        git_push_report = push_repo_run_to_github(
            ROOT,
            RUN_ID,
            run_relative_dir=REPO_RUN_DIR.relative_to(ROOT),
            remote_name=effective_params.get("AUTO_PUSH_REMOTE_NAME", AUTO_PUSH_REMOTE_NAME),
            branch=effective_params.get("AUTO_PUSH_BRANCH", AUTO_PUSH_BRANCH),
            print_fn=print,
        )
    except (RuntimeError, subprocess.CalledProcessError) as exc:
        print(f"[GIT] Auto-push skipped: {exc}")
        git_push_report = {"enabled": True, "pushed": False, "run_dir": str(REPO_RUN_DIR), "error": str(exc)}
else:
    git_push_report = {"enabled": False, "pushed": False, "run_dir": str(REPO_RUN_DIR)}
STATE["git_push_report"] = git_push_report
disconnect_report = build_notebook_completion_report(
    state=STATE,
    telemetry=TELEMETRY,
    repo_run_exports=REPO_RUN_EXPORTS,
    notebook_export_path=notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH,
)
STATE["auto_disconnect_report"] = disconnect_report
print(f"[COLAB] completion checks -> {disconnect_report['checks']}")
maybe_auto_disconnect_colab_runtime(
    enabled=bool(effective_params.get("AUTO_DISCONNECT_RUNTIME", AUTO_DISCONNECT_RUNTIME)),
    grace_period_sec=int(effective_params.get("AUTO_DISCONNECT_GRACE_SECONDS", AUTO_DISCONNECT_GRACE_SECONDS)),
    telemetry=TELEMETRY,
    completion_report=disconnect_report,
)


[DOGRULAMA (referans)] ornek=223 sinif=5 accuracy=0.8027 ood_auroc=0.9196 sure_ds_f1=0.2712 conformal_cov=0.9552
[TEST (ayrilmis)] ornek=223 sinif=5 accuracy=0.8117 ood_auroc=0.9172 sure_ds_f1=0.3048 conformal_cov=0.9731
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.
[OPT] mode=continue status=active eligible=0 frontier=0 executed=0
[OPT] next_proposal rank=1 signature={"training.adapt
